# S² (order parameter) vs prediction quality

Per protein: Kendall's τ vs mean S². Per residue: |error| vs S².

In [ ]:
import os
os.makedirs("Figs", exist_ok=True)  # figures are written to ./Figs

def set_style(base=13, family="Arial"):
    import matplotlib.pyplot as plt
    plt.rcParams.update({
        "font.family": family,
        "font.size": base,
        "axes.titlesize": base + 2,
        "axes.labelsize": base,
        "xtick.labelsize": base - 5,
        "ytick.labelsize": base - 2,
        "legend.fontsize": base - 2,
        "svg.fonttype": "none",
    })
set_style()

In [ ]:
import json
import numpy as np, pandas as pd
from scipy.stats import kendalltau

FIG4 = "../../../data/analysis/Fig4"
AF   = f"{FIG4}/all_features"
GTV5 = "../../../data/rBMRB/cluster_30p/PANAV/test.json"
FIG  = "Figs"

# ---- RECOMPUTE abserr + per-protein Kendall's tau from the v5 seed-ensemble predictions ----
# Same recipe as Grouping_geom.ipynb: "mean"/"mask" are the 10-seed ensemble mean and its own
# validity mask (tag/outlier exclusion is already folded into "mask" upstream -- do NOT re-apply
# a separate per-protein mean-centering or 4sigma-BMRB gate on top of it).
#   - pred/mask arrays are right-padded to the batch's longest sequence, so truncate to len(gt_seq)
#     instead of skipping the whole protein when longer; only skip if actually shorter than gt_seq.
pred = json.load(open(f"{FIG4}/results/predictions_N_seed_ensemble_10seed.json"))
gt   = json.load(open(GTV5))
mean_d, mask_d = pred["mean"], pred["mask"]
key_of = {v: k for k, v in gt["entry_ID"].items()}

ae_rows = []
proteins, tau_per_prot = [], []
length_mismatch = 0
for e, key in key_of.items():
    if e not in mean_d:
        continue
    seq, Nv = gt["sequence"][key], gt["N"][key]
    n = len(seq)
    pred_arr_full = np.asarray(mean_d[e], dtype=object)
    mask_full = np.asarray(mask_d[e], dtype=bool)
    if len(pred_arr_full) < n or len(mask_full) < n:
        length_mismatch += 1
        continue
    pm, msk = pred_arr_full[:n], mask_full[:n]

    base = np.array([r != "P" for r in seq]) & np.array([g is not None for g in Nv]) & msk
    if base.sum() == 0:
        continue
    t = np.array(Nv, dtype=object)[base].astype(float); pf = pm[base].astype(float)
    fidx = np.where(base)[0]
    err = abs(pf - t)

    proteins.append(e)
    tau_per_prot.append(kendalltau(pf, t).statistic)

    for fi, a in zip(fidx, err):
        ae_rows.append((e, int(fi) + 1, float(a)))

ae = pd.DataFrame(ae_rows, columns=["entry", "resnum", "abserr"])
tau_per_prot = np.array(tau_per_prot)
print(f"abserr | rows={len(ae)} | proteins={ae.entry.nunique()} (of {len(key_of)} entries in the v5 test set, "
      f"{length_mismatch} skipped for pred shorter than gt)")
print(f"Proteins evaluated for tau: {len(proteins)}")
print(f"Mean Kendall's tau = {tau_per_prot.mean():.4f} ± {tau_per_prot.std():.4f}")

In [ ]:
# ---- per-residue S2 from all_features/per_residue_records_full.json (same source as Grouping_geom.ipynb) ----
prr = json.load(open(f"{AF}/per_residue_records_full.json"))
s2_rows = [(e, r["resnum"], r["s2"]) for e, rl in prr.items() for r in rl if pd.notna(r.get("s2"))]
s2_df = pd.DataFrame(s2_rows, columns=["entry", "resnum", "s2"])

df = ae.merge(s2_df, on=["entry", "resnum"], how="inner")
df = df[df["s2"].notna()]   # belt-and-suspenders: drop any NaN s2 that slipped through the merge
print(f"S2 x abserr | rows={len(df)} | proteins={df.entry.nunique()} (of {ae.entry.nunique()} proteins with abserr)")

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr

# ---------- Panel 2: per-residue S2 vs |error| ----------
res_s2     = df["s2"].to_numpy()
res_abserr = df["abserr"].to_numpy()

r_r, p_r = pearsonr(res_s2, res_abserr)
rho_r, _ = spearmanr(res_s2, res_abserr)

# binned mean |error| in S2 bins, for a clear trend over the cloud
bins = np.linspace(0, 1, 11)
bin_idx = np.digitize(res_s2, bins)
centers, means, sems = [], [], []
for k in range(1, len(bins)):
    sel = bin_idx == k
    if sel.sum() >= 5:
        centers.append((bins[k - 1] + bins[k]) / 2)
        means.append(res_abserr[sel].mean())
        sems.append(res_abserr[sel].std() / np.sqrt(sel.sum()))

fig, ax = plt.subplots(figsize=(6.5, 5.2))
# ax.scatter(res_s2, res_abserr, alpha=0.12, s=8, color="#61AFEF", edgecolors="none")
ax.errorbar(centers, means, yerr=sems, marker="o", ms=6, lw=2, capsize=4,
            color="#56B6C2", label="binned mean ± SEM")
ax.set_xlabel("Residue S2 (order parameter)")
ax.set_ylabel("|error| (ppm)")
ax.set_ylim(0.5, 3)
# ax.set_title(f"Pearson r = {r_r:.3f}, Spearman ρ = {rho_r:.3f}  (n = {len(res_s2)})")
ax.legend(frameon=False)
# ax.grid(alpha=0.3, linestyle="--")
plt.tight_layout()
plt.savefig(
    "Figs/s2_vs_error_perresidue.png",
    dpi=300, bbox_inches="tight",
)
plt.show()
print(f"Per-residue  n={len(res_s2)}  Pearson r={r_r:.3f} (p={p_r:.2e})  Spearman ρ={rho_r:.3f}")
print("Binned mean |error| (ppm) by S2 bin:")
for c, mu in zip(centers, means):
    print(f"  S2 ~{c:.2f}: {mu:.3f} ppm")

In [ ]:
# ---------- binned BOX plot of |error| (style from error_vs_p_exchange_perSS_noHIS_4pub) ----------
set_style(base=16)

_x   = np.asarray(res_s2)
_y   = np.asarray(res_abserr)
_bins = np.linspace(0, 1, 11)
_idx  = np.digitize(_x, _bins)

_data, _labels, _centers = [], [], []
for k in range(1, len(_bins)):
    sel = _idx == k
    if sel.sum() >= 5:
        _data.append(_y[sel])
        _centers.append((_bins[k - 1] + _bins[k]) / 2)
        _labels.append("{:.1f}–{:.1f}\n(n={})".format(_bins[k - 1], _bins[k], int(sel.sum())))

_cmap = plt.get_cmap("Blues")
_cols = [_cmap(0.25 + 0.6 * i / max(1, len(_data) - 1)) for i in range(len(_data))]

fig, ax = plt.subplots(figsize=(8, 5.5))
bp = ax.boxplot(_data, showfliers=False, patch_artist=True, medianprops=dict(color="black"))
for patch, col in zip(bp["boxes"], _cols):
    patch.set_facecolor(col)
ax.set_xticks(range(1, len(_data) + 1))
ax.set_xticklabels(_labels)
ax.set_xlabel("S\u00b2 (order parameter)")
ax.set_ylabel("|error| (ppm)")
ax.set_ylim(-0.1, 7)
ax.set_title("Per-residue |error| by S\u00b2 (order parameter) bin (n = {})".format(len(_x)))
# ax.grid(alpha=0.3, ls="--", axis="y")
plt.tight_layout()
plt.savefig("Figs/s2_vs_error_binbox.svg", format="svg", bbox_inches="tight")
plt.show()
print("Binned box plot saved: s2_vs_error_binbox.svg")
for c, d in zip(_centers, _data):
    print(f"  bin ~{c:.2f}: median={np.median(d):.3f} ppm  IQR=[{np.percentile(d,25):.3f}, {np.percentile(d,75):.3f}]  n={len(d)}")